# NeuralZip: storage, setup reuse, and GPU inference

This notebook presents the complete public API. The first section is the shortest path: load an ordinary Hugging Face checkpoint, compress it, and use the returned native model for inference. The later sections expose setup, encoding, decoding, timing, and transfer separately.

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from neuralzip import NeuralZip

model_id = "Qwen/Qwen2.5-0.5B"

## One-call compressed inference

`from_pretrained()` loads BF16 weights on CPU, fits a GPU-compatible setup when `setup=None`, compresses eligible weights, places the resident state, and returns the ordinary Transformers model with a NeuralZip runtime attached.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
started = time.perf_counter()
model = NeuralZip.from_pretrained(model_id, device="cuda:0")
load_and_compress_seconds = time.perf_counter() - started

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=model.neuralzip_runtime.device,
)
output = generator(
    "Machine learning is", max_new_tokens=20, do_sample=False
)
print(f"Load + setup + compression: {load_and_compress_seconds:.3f} s")
print(output)

## Persistent compressed checkpoint

Persisting the attached model avoids reloading and recompressing the original BF16 checkpoint in another process. The saved resident tensors use Safetensors; the compressed export and manifests are versioned and verified on load.

In [ ]:
NeuralZip.save_pretrained(
    model,
    "qwen-neuralzip",
    tokenizer=tokenizer,
)

# A future process can load the compressed checkpoint directly.
del generator, model
torch.cuda.empty_cache()
loaded_model = NeuralZip.load_pretrained(
    "qwen-neuralzip", device="cuda:0"
)

## Manual setup, encode, and decode

Use the explicit setup API when each phase must be measured separately. `NeuralZip` is stateless: `fit()` returns an immutable setup, `encode()` receives that setup explicitly and returns the compressed value plus optional timing statistics, and `decode()` reconstructs a fresh model bit-for-bit without needing the setup. When no loaded model is needed, `fit()` can also receive a Hugging Face model ID directly, for example `NeuralZip.fit("openai-community/gpt2")`.

In [ ]:
source_model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=torch.bfloat16
).eval()

setup = NeuralZip.fit(source_model, threads=4)
encoded, encode_stats = NeuralZip.encode(
    source_model, setup=setup, threads=4, return_stats=True
)
restored, decode_stats = NeuralZip.decode(
    encoded, threads=4, return_stats=True
)

print(f"Ratio: {encoded.compression_ratio:.4f}")
print(f"Setup: {setup.setup_report.setup_seconds:.3f} s")
print(f"Encode: {encode_stats.encode_seconds:.3f} s")
print(f"Decode: {decode_stats.decode_seconds:.3f} s")

In [ ]:
# Verify an exact floating-point round trip.
for original, recovered in zip(
    source_model.parameters(), restored.parameters(), strict=True
):
    assert torch.equal(
        original.detach().cpu().view(torch.uint8),
        recovered.detach().cpu().view(torch.uint8),
    )
print("Bitwise-exact round trip")

## Reuse or transfer the setup

The setup can be persisted independently and loaded wherever it is needed. A compatible target is mapped from metadata only, so clustering and Huffman construction stay outside its encoding path. Setup files use pickle and must only come from trusted sources.

In [ ]:
NeuralZip.save_setup(setup, "qwen-family.setup.pkl")

target_model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=torch.bfloat16
).eval()
reused_setup = NeuralZip.load_setup("qwen-family.setup.pkl")
mapping = NeuralZip.map_target(target_model, setup=reused_setup)
encoded_target, target_stats = NeuralZip.encode(
    target_model, setup=reused_setup, threads=4, return_stats=True
)

print(f"Mapped: {mapping['matched_tensors']}/{mapping['target_tensors']} tensors")
print(f"Mapping: {target_stats.mapping_seconds:.3f} s")
print(f"Encode: {target_stats.encode_seconds:.3f} s")